In [14]:
import pandas as pd
import nest_asyncio
from vllm import LLM, SamplingParams

In [19]:
dados = pd.read_csv('../dados/chamados_com_predicoes.csv', index_col='id_chamado')

In [39]:
nest_asyncio.apply()

llm = LLM(
    model="Qwen/Qwen3.5-9B",
    dtype="float16",                  
    gpu_memory_utilization=0.85,      
    max_model_len=1024,               
    max_num_seqs=64,                  
    enforce_eager=True                
)

sampling_params = SamplingParams(
    temperature=0.0,  # Resposta determinística
    max_tokens=25,  # Textos curtos de classificação não precisam de mais que isso
    stop=[
        "<|im_end|>",
        "<|endoftext|>",
    ], 
)

textos_para_classificar = dados["texto"].tolist()
prompts_formatados = []

for texto in textos_para_classificar:
    prompt = f"""<|im_start|>system
    Você é um classificador direto de segurança. Avalie se o texto envolve brigas, assaltos, choques elétricos ou atropelamentos.
    Responda APENAS com "Sim" ou "Não" logo após fechar a tag de pensamento.<|im_end|>
    <|im_start|>user
    Texto: "{texto}"<|im_end|>
    <|im_start|>assistant
    <think>"""
    prompts_formatados.append(prompt)

print(f"Iniciando a classificação em lote de {len(prompts_formatados)} textos...")
outputs = llm.generate(prompts_formatados, sampling_params)

# 7. Processando os resultados rapidamente
resultados = []
for output in outputs:
    resposta_limpa = output.outputs[0].text.split("</think>")[-1].strip()
    resultados.append(resposta_limpa)

print("Processamento concluído!")
print(f"Exemplo de resultado: {resultados[0]}")

INFO 07-07 19:00:21 [api_utils.py:273] non-default args: {'dtype': 'float16', 'max_model_len': 1024, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3.5-9B'}
INFO 07-07 19:00:22 [model.py:598] Resolved architecture: Qwen3_5ForConditionalGeneration
WARNING 07-07 19:00:22 [model.py:2063] Casting torch.bfloat16 to torch.float16.
INFO 07-07 19:00:22 [model.py:1725] Using max model len 1024
WARNING 07-07 19:00:22 [vllm.py:1062] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 07-07 19:00:22 [vllm.py:1110] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 07-07 19:00:22 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 07-07 19

(EngineCore pid=33383) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=33383) INFO 07-07 19:00:32 [parallel_state.py:1588] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://192.168.1.25:56807 backend=nccl
(EngineCore pid=33383) INFO 07-07 19:00:32 [parallel_state.py:1923] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=33383) INFO 07-07 19:00:33 [topk_topp_sampler.py:55] Using FlashInfer for top-p & top-k sampling.


(EngineCore pid=33383) [transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=33383) INFO 07-07 19:00:43 [gpu_model_runner.py:5160] Starting to load model Qwen/Qwen3.5-9B...
(EngineCore pid=33383) INFO 07-07 19:00:43 [cuda.py:539] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=33383) INFO 07-07 19:00:43 [mm_encoder_attention.py:373] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=33383) INFO 07-07 19:00:43 [qwen_gdn_linear_attn.py:228] Using Triton/FLA GDN prefill kernel (requested=auto, head_k_dim=128).
(EngineCore pid=33383) INFO 07-07 19:00:43 [cuda.py:480] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=33383) INFO 07-07 19:00:43 [flash_attn.py:670] Using FlashAttention version 2
(EngineCore pid=33383) ERROR 07-07 19:00:44 [gpu_model_runner.py:5253] Failed to load model - not enough GPU memory. Try lowering --gpu-memory-utilization to free memory for weights, increasing --tensor-parallel-s

(EngineCore pid=33383) Process EngineCore:
(EngineCore pid=33383) Traceback (most recent call last):
(EngineCore pid=33383)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=33383)     self.run()
(EngineCore pid=33383)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=33383)     self._target(*self._args, **self._kwargs)
(EngineCore pid=33383)   File "/home/saviobarreto/Documents/desafio-cientista-dados-junior-ia-pref/vllm_env/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1235, in run_engine_core
(EngineCore pid=33383)     raise e
(EngineCore pid=33383)   File "/home/saviobarreto/Documents/desafio-cientista-dados-junior-ia-pref/vllm_env/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1200, in run_engine_core
(EngineCore pid=33383)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=33383)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [35]:
resultados

['Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',
 'Sim',
 'Não',
 'Não',
 'Não',
 'Não',
 'Não',


In [36]:
from collections import Counter

# Passa a sua lista de resultados direto no Counter
distribuicao = Counter(resultados)

print("Distribuição dos resultados:")
for classe, quantidade in distribuicao.items():
    porcentagem = (quantidade / len(resultados)) * 100
    print(f"- {classe}: {quantidade} ({porcentagem:.2f}%)")

Distribuição dos resultados:
- Não: 4909 (98.18%)
- Sim: 84 (1.68%)
- Thinking Process:

1.  **Analyze the Request:**
    *   Role: Direct safety classifier.: 7 (0.14%)


In [38]:
del llm

(EngineCore pid=32438) INFO 07-07 19:00:10 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=32438) INFO 07-07 19:00:10 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=32438) INFO 07-07 19:00:10 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=32438) INFO 07-07 19:00:10 [core.py:1227] [shutdown] EngineCore: exiting busy loop


In [29]:
outputs[0].outputs[0].text

'<think>\nThinking Process:\n\n1.  **'